# Class 2 practice — Chicago Taxi Trips: exploring data & calculating market shares

Run this on the Yale HPC (Open OnDemand → JupyterLab), from your home directory. Needs
the `esai_2026` course folder symlinked in (`~/esai_2026`) and `pandas` installed.

**Note on quotes**: if you ever get `SyntaxError: invalid character '‘' (U+2018)`, it means
a quote mark got auto-converted to a "smart quote" by whatever you copied it from — retype
the straight quotes by hand.

## 0. Install packages (run once per fresh session)

In [ ]:
import sys
!{sys.executable} -m pip install --user pandas matplotlib

## 1. Load the data

Start with 1,000 rows only — the full dataset is large; get the code working first.

In [ ]:
import pandas as pd

DATA_PATH = '~/esai_2026/data/chicago_taxi_trips_2024/Taxi_Trips_2024.csv'
df = pd.read_csv(DATA_PATH, nrows=1000)

## 2. Explore

Column names with spaces (e.g. `Trip Total`) break dot notation — `df.Trip Total.max()` is a
`SyntaxError`. Use bracket notation: `df["Trip Total"]`.

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.iloc[0]

In [ ]:
df.describe()

## 3. Derived column: speed (mph)

Zero-second trips produce `inf` when we divide — that's a data-quality issue in the source,
not a bug in this code. We filter it out in the cleaning step below rather than silently.

In [ ]:
df["mph"] = df["Trip Miles"] / (df["Trip Seconds"] / 60**2)

print("max mph, unfiltered (includes inf from 0-second trips):", df["mph"].max())
print("max mph, trips with duration > 0:", df.loc[df["Trip Seconds"] > 0, "mph"].max())

## 4. Clean

Filter to trips that actually happened: positive duration, positive distance, positive fare.
Store the cleaned result as `df1` so the original `df` is preserved for comparison.

In [ ]:
condition1 = df["Trip Seconds"] > 0
condition2 = df["Trip Miles"] > 0
condition3 = df["Fare"] > 0

df1 = df[condition1 & condition2 & condition3]

print(f"{len(df)} rows -> {len(df1)} rows after cleaning ({len(df) - len(df1)} dropped)")

## 5. Market shares

Two definitions — by trip count and by revenue — as separate functions, since the professor
flagged the definition isn't obvious and they can diverge (a company with fewer, higher-fare
trips can have a bigger revenue share than trip-count share).

In [ ]:
def market_share_by_count(data, firm_col, year_col=None, year=None):
    """Market share by number of trips: trips for firm / total trips."""
    if year is not None:
        data = data[data[year_col] == year]
    return data[firm_col].value_counts(normalize=True)


def market_share_by_revenue(data, firm_col, revenue_col, year_col=None, year=None):
    """Market share by revenue: firm's total revenue / total revenue in the sample."""
    if year is not None:
        data = data[data[year_col] == year]
    return data.groupby(firm_col)[revenue_col].sum() / data[revenue_col].sum()

### Verify with synthetic data before trusting it on the real thing

Exactly what the professor called out: build fake data with a known, hand-computed answer,
and confirm the function recovers it — don't just trust that the code (yours or an LLM's) is
right because it ran without an error.

In [ ]:
fake = pd.DataFrame({
    "Company": ["A", "A", "A", "B", "B", "C", "C", "C", "C"],  # A: 3, B: 2, C: 4 -> 9 trips
    "Fare": [10, 10, 10, 50, 50, 5, 5, 5, 5],                    # A: $30, B: $100, C: $20 -> $150
})

shares_by_count = market_share_by_count(fake, "Company")
shares_by_revenue = market_share_by_revenue(fake, "Company", "Fare")

# known answers, computed by hand
expected_count = {"A": 3 / 9, "B": 2 / 9, "C": 4 / 9}
expected_revenue = {"A": 30 / 150, "B": 100 / 150, "C": 20 / 150}

for firm, expected in expected_count.items():
    assert abs(shares_by_count[firm] - expected) < 1e-9, f"count share wrong for {firm}"
for firm, expected in expected_revenue.items():
    assert abs(shares_by_revenue[firm] - expected) < 1e-9, f"revenue share wrong for {firm}"

# shares must always sum to 1
assert abs(shares_by_count.sum() - 1) < 1e-9
assert abs(shares_by_revenue.sum() - 1) < 1e-9

print("synthetic-data checks passed")
print(shares_by_count)
print(shares_by_revenue)

### Run it on the real (cleaned, 2024) data

Adjust `firm_col` / `year_col` to match the actual column names once you've checked
`df1.columns` — taxi datasets commonly use `Company` and a trip-start-time column you'd
need to convert to a year first.

In [ ]:
# df1["year"] = pd.to_datetime(df1["Trip Start Timestamp"]).dt.year
# by_count = market_share_by_count(df1, "Company", year_col="year", year=2024)
# by_revenue = market_share_by_revenue(df1, "Company", "Fare", year_col="year", year=2024)
# by_count.sort_values(ascending=False).head(10)